In [ ]:
import io
import shutil
import zipfile
import requests
import pandas as pd
from pathlib import Path
import os
import glob
import json
import math
import sys
import time
from pathlib import Path
import torch
import re
import whisper
from transformers import AutoProcessor, AutoModelForImageTextToText

In [ ]:
WHISPER_MODEL_NAME = "medium"
WHISPER_LANGUAGE = None
WHISPER_TASK = "transcribe"

QWEN_MODEL = "Qwen/Qwen3-VL-8B-Instruct"
QWEN_MAX_NEW_TOKENS = 256

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

In [ ]:
def init_models():
    print(f"1. Loading OpenAI Whisper: {WHISPER_MODEL_NAME}")

    whisper_model = whisper.load_model(
        WHISPER_MODEL_NAME,
        device=DEVICE
    )

    print(f"2. Loading Qwen: {QWEN_MODEL}")

    qwen_processor = AutoProcessor.from_pretrained(
        QWEN_MODEL
    )

    qwen_model = AutoModelForImageTextToText.from_pretrained(
        QWEN_MODEL,
        torch_dtype=(
            torch.float16
            if DEVICE == "cuda"
            else torch.float32
        ),
        device_map="auto"
    )

    return (
        whisper_model,
        qwen_processor,
        qwen_model
    )

In [ ]:
QWEN_PROMPT_TEMPLATE = (
    "Bạn là bộ hậu xử lý văn bản ASR tiếng Việt.\n\n"

    "Văn bản dưới đây là kết quả nhận dạng giọng nói tự động (ASR), "
    "có thể chứa lỗi chính tả, thiếu dấu, viết sai từ, viết sai tên riêng "
    "hoặc ngắt câu không chính xác.\n\n"

    "Nhiệm vụ của bạn là CHỈ sửa các lỗi nhận dạng rõ ràng "
    "để văn bản đúng và tự nhiên hơn, đồng thời PHẢI giữ nguyên nội dung gốc.\n\n"

    "QUY TẮC BẮT BUỘC:\n"
    "1. Không thêm bất kỳ thông tin nào không có trong văn bản gốc.\n"
    "2. Không suy diễn hoặc đoán nội dung bị thiếu.\n"
    "3. Không viết lại hoặc diễn đạt lại câu nếu không cần thiết.\n"
    "4. Không thay đổi ý nghĩa của câu.\n"
    "5. Không thêm tên người, địa danh, tổ chức, thời gian, số liệu "
    "hoặc sự kiện nếu chúng không xuất hiện trong văn bản gốc.\n"
    "6. Giữ nguyên các con số, tên riêng và thuật ngữ nếu không có lỗi rõ ràng.\n"
    "7. Chỉ sửa lỗi chính tả, dấu câu, dấu tiếng Việt và lỗi ngắt từ rõ ràng.\n"
    "8. Nếu không chắc chắn một từ có bị nhận dạng sai hay không, "
    "hãy GIỮ NGUYÊN từ đó.\n"
    "9. Không giải thích, không nhận xét, không tóm tắt.\n"
    "10. Chỉ trả về phiên bản văn bản đã sửa, không thêm bất kỳ nội dung nào khác.\n\n"

    "VĂN BẢN GỐC:\n"
    "{raw_text}"
)


def correct_text_with_qwen(
    raw_text: str,
    processor,
    model
) -> str:
    """
    Chỉ dùng Qwen để sửa text trong bộ nhớ.
    Kết quả cuối cùng vẫn ghi vào field 'text' của schema JSON cũ.
    """
    if not raw_text.strip():
        return raw_text

    prompt = QWEN_PROMPT_TEMPLATE.format(
        raw_text=raw_text
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    text_input = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text_input],
        return_tensors="pt",
        padding=True
    )

    if hasattr(model, "device"):
        inputs = {
            key: value.to(model.device)
            if hasattr(value, "to")
            else value
            for key, value in inputs.items()
        }

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=QWEN_MAX_NEW_TOKENS
        )

    input_ids = inputs.get("input_ids")

    if input_ids is not None:
        generated_ids = [
            output_ids[len(input_ids[0]):]
            for output_ids in generated_ids
        ]

    corrected_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    # Nếu model trả về rỗng hoặc bất thường,
    # giữ nguyên Whisper text để tránh mất dữ liệu.
    if not corrected_text:
        return raw_text

    return corrected_text

In [ ]:
import math

def whisper_transcribe(
    video_path: str,
    whisper_model,
    qwen_processor,
    qwen_model
):
    """
    Video -> Whisper segments -> Qwen correction.

    JSON schema không đổi:
    {
        "start_ms": ...,
        "end_ms": ...,
        "text": ...,
        "confidence": ...
    }
    """
    result = whisper_model.transcribe(
        video_path,
        language=WHISPER_LANGUAGE,
        task=WHISPER_TASK,
        verbose=False,
        fp16=(DEVICE == "cuda"),
        condition_on_previous_text=True
    )

    segments = []
    raw_segments = result.get("segments", [])

    for index, seg in enumerate(raw_segments, start=1):
        raw_text = str(seg.get("text", "")).strip()

        if not raw_text:
            continue

        start_ms = int(round(float(seg["start"]) * 1000))
        end_ms = int(round(float(seg["end"]) * 1000))

        # Whisper confidence xấp xỉ.
        avg_logprob = float(seg.get("avg_logprob", -1.0))
        no_speech_prob = float(seg.get("no_speech_prob", 0.0))

        confidence = (
            math.exp(avg_logprob)
            * max(0.0, 1.0 - no_speech_prob)
        )

        confidence = max(0.0, min(1.0, confidence))

        # Qwen chỉ sửa text.
        # Không thay đổi timestamp hay schema.
        cleaned_text = correct_text_with_qwen(
            raw_text,
            qwen_processor,
            qwen_model
        )

        segments.append({
            "start_ms": start_ms,
            "end_ms": end_ms,
            "text": cleaned_text,
            "confidence": round(confidence, 4)
        })

    return segments

In [ ]:
def process_video_asr(
    video_path: str,
    output_folder: str,
    whisper_model,
    qwen_processor,
    qwen_model
):
    video_id = Path(
        video_path
    ).stem

    video_output_folder = os.path.join(
        output_folder,
        video_id
    )

    os.makedirs(
        video_output_folder,
        exist_ok=True
    )

    segments = whisper_transcribe(
        video_path,
        whisper_model,
        qwen_processor,
        qwen_model
    )

    result = {
        "video_id": video_id,
        "segments": segments
    }

    output_json_path = os.path.join(
        video_output_folder,
        f"{video_id}.json"
    )

    with open(
        output_json_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"Saved: {output_json_path}"
    )

    return result

In [ ]:
def find_and_check_csv():
    if os.path.exists('/kaggle/working'):
        print("Đang chạy trên môi trường Kaggle...")
        csv_path = '/kaggle/input/datasets/dipthnnguyn/bactch1/Batch1.csv' 
        output_dir = '/kaggle/working/dataset_unzipped'
    else:
        print("Đang chạy trên máy tính cá nhân (Local)...")
        csv_path = './Batch1.csv' 
        output_dir = './dataset_unzipped'
    os.makedirs(output_dir, exist_ok=True)
    try:
        df = pd.read_csv(csv_path)
        print(f"Đã nạp file CSV thành công. Tổng số dòng: {len(df)}")
        # In ra danh sách các cột để bạn đối chiếu xem đã gõ đúng tên cột chứa link video chưa
        print(f"Các cột hiện có trong file CSV: {list(df.columns)}")
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file CSV tại {csv_path}")
        df = pd.DataFrame() 
    return output_dir

In [ ]:
def load_completed_zips(log_file_path: str) -> set:
    """Đọc danh sách các zip đã xử lý xong từ file log."""
    if os.path.exists(log_file_path):
        with open(log_file_path, "r", encoding="utf-8") as f:
            return set(line.strip() for line in f if line.strip())
    return set()


def mark_zip_as_completed(log_file_path: str, zip_filename: str):
    """Ghi nhận một file zip đã hoàn thành vào file log."""
    with open(log_file_path, "a", encoding="utf-8") as f:
        f.write(f"{zip_filename}\n")


def is_video_allowed(video_path: str) -> bool:
    """
    Kiểm tra xem file video có nằm trong dải cho phép theo image_6b4184.png không.
    Dải cho phép:
    - L22_V028 - L22_V031
    - L25_V037 - L25_V088
    - L26_V265 - L26_V299
    - L26_V454 - L26_V499
    """
    filename = os.path.basename(video_path).upper()

    match = re.search(r'(L\d{2}_V\d{3})', filename)
    if not match:
        return False

    vid_id = match.group(1)
    prefix = vid_id[:5]
    num = int(vid_id[5:])

    if prefix == 'L22_V':
        return 28 <= num <= 31
    elif prefix == 'L25_V':
        return 37 <= num <= 88
    elif prefix == 'L26_V':
        return (454 <= num <= 499)

    return False


def cleanup_video_dir(output_dir: str):
    """
    Xóa TOÀN BỘ thư mục chứa video tạm rồi tạo lại.
    Không đụng vào json_dir hoặc completed_zips.txt.
    """
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir, ignore_errors=True)

    os.makedirs(output_dir, exist_ok=True)


def asr_processing_videos(
    df, output_dir, json_dir, max_hours: float = 15.0, log_file_path: str = None
):
    START_TIME = time.time()

    completed_zips = set()
    if log_file_path:
        completed_zips = load_completed_zips(log_file_path)
        print(
            f"-> Đã tìm thấy {len(completed_zips)} gói zip đã hoàn thành từ trước."
        )

    print(" - Đang khởi tạo mô hình ASR (Whisper + Qwen)...")
    whisper_model, qwen_processor, qwen_model = init_models()

    df.columns = df.columns.str.strip()
    VIDEO_COLUMN_NAME = "Filenames"
    VIDEO_URL_COLUMN = "Download link"

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(json_dir, exist_ok=True)

    if (
        not df.empty
        and VIDEO_COLUMN_NAME in df.columns
        and VIDEO_URL_COLUMN in df.columns
    ):
        print(
            "Đã tìm thấy các cột dữ liệu hợp lệ, bắt đầu duyệt qua tất cả các gói zip..."
        )

        for index, row in df.iterrows():
            elapsed_hours = (time.time() - START_TIME) / 3600

            if elapsed_hours >= max_hours:
                print(
                    f"\n[TIMEOUT GUARD] Đã chạy {elapsed_hours:.2f}h (vượt giới hạn "
                    f"{max_hours}h)."
                )
                print("Đang dừng an toàn để lưu Output...")
                cleanup_video_dir(output_dir)
                sys.exit(0)

            filename_val = str(row[VIDEO_COLUMN_NAME]).strip()
            zip_link = str(row[VIDEO_URL_COLUMN]).strip()

            if (
                pd.isna(row[VIDEO_URL_COLUMN])
                or zip_link == "nan"
                or not zip_link.startswith("http")
            ):
                continue
            
            if not (
                filename_val.lower().startswith("video")
                and filename_val.lower().endswith(".zip")
            ):
                continue

            target_package_dir = os.path.join(
                json_dir,
                filename_val.replace(".zip", "")
            )
            os.makedirs(target_package_dir, exist_ok=True)

            print(
                f"\n[{index + 1}/{len(df)}] BẮT ĐẦU XỬ LÝ GÓI VIDEO: {filename_val}"
            )

            local_zip_path = os.path.join(
                output_dir,
                f"temp_{filename_val}"
            )

            try:
                # Luôn bắt đầu mỗi ZIP với thư mục video sạch.
                # Nếu notebook bị dừng ở ZIP trước, video cũ cũng không bị trộn vào ZIP mới.
                cleanup_video_dir(output_dir)

                print(
                    " - Đang tải và bung nén qua HTTP "
                    "(Lưu thẳng xuống ổ cứng)..."
                )

                max_retries = 3
                download_success = False

                for attempt in range(1, max_retries + 1):
                    try:
                        with requests.get(
                            zip_link,
                            stream=True,
                            timeout=120
                        ) as response:
                            if response.status_code != 200:
                                print(
                                    f"   ! Lỗi tải file (mã {response.status_code}). "
                                    f"Thử lại lần {attempt}/{max_retries}..."
                                )
                                time.sleep(3)
                                continue

                            with open(local_zip_path, "wb") as f:
                                for chunk in response.iter_content(chunk_size=8192):
                                    if chunk:
                                        f.write(chunk)

                        # Giải nén từ file ZIP tạm trên ổ cứng.
                        with zipfile.ZipFile(local_zip_path, "r") as z:
                            z.extractall(output_dir)

                        # ZIP tạm không còn cần thiết sau khi extract.
                        if os.path.exists(local_zip_path):
                            os.remove(local_zip_path)

                        download_success = True
                        break

                    except Exception as req_e:
                        print(
                            f"   ! Gián đoạn mạng/lỗi tải file ({req_e}). "
                            f"Thử lại lần {attempt}/{max_retries}..."
                        )

                        if os.path.exists(local_zip_path):
                            try:
                                os.remove(local_zip_path)
                            except Exception:
                                pass

                        time.sleep(5)

                if not download_success:
                    print(
                        f" - BỎ QUA GÓI NÀY do mạng chập chờn sau "
                        f"{max_retries} lần thử."
                    )
                    continue

                video_extensions = ['*.mp4', '*.avi', '*.mkv']
                all_video_files = []

                for ext in video_extensions:
                    all_video_files.extend(
                        glob.glob(
                            os.path.join(output_dir, "**", ext),
                            recursive=True
                        )
                    )

                if not all_video_files:
                    print(" - Không tìm thấy file video nào trong gói này.")
                    continue

                allowed_video_files = [
                    f for f in all_video_files
                    if is_video_allowed(f)
                ]

                if not allowed_video_files:
                    print(
                        " - Không có video nào trong gói này nằm trong dải "
                        "quy định. Bỏ qua chạy ASR."
                    )
                else:
                    video_files = sorted(set(allowed_video_files))

                    print(
                        f" * Tìm thấy {len(video_files)} video ĐƯỢC PHÉP CHẠY. "
                        "Bắt đầu ASR..."
                    )

                    for vid_index, vid_path in enumerate(video_files, start=1):
                        print(
                            f"   + [{vid_index}/{len(video_files)}] Xử lý: "
                            f"{os.path.basename(vid_path)}"
                        )

                        process_video_asr(
                            video_path=vid_path,
                            output_folder=target_package_dir,
                            whisper_model=whisper_model,
                            qwen_processor=qwen_processor,
                            qwen_model=qwen_model,
                        )

                        elapsed_hours = (time.time() - START_TIME) / 3600

                        if elapsed_hours >= max_hours:
                            print(
                                f"\n[TIMEOUT GUARD] Đạt mốc {elapsed_hours:.2f}h "
                                "khi đang chạy."
                            )
                            print(
                                "Đang dọn dẹp toàn bộ video tạm "
                                "của ZIP hiện tại..."
                            )
                            cleanup_video_dir(output_dir)
                            print("Dừng hệ thống an toàn.")
                            sys.exit(0)

                # ZIP đã chạy đến hết logic xử lý.
                # Đánh dấu completed trước khi chuyển sang ZIP tiếp theo.
                if log_file_path:
                    mark_zip_as_completed(log_file_path, filename_val)
                    completed_zips.add(filename_val)
                    print(
                        f" -> Đã lưu '{filename_val}' vào danh sách hoàn tất."
                    )

            except Exception as e:
                print(
                    f" - Lỗi khi xử lý {filename_val}: {e}"
                )

            finally:
                # Đây là cleanup QUAN TRỌNG NHẤT:
                # dù thành công, lỗi, continue hay timeout,
                # video và ZIP tạm của ZIP hiện tại luôn được xóa.
                try:
                    if os.path.exists(local_zip_path):
                        os.remove(local_zip_path)
                except Exception as cleanup_zip_error:
                    print(
                        f"   ! Không thể xóa file ZIP tạm "
                        f"{local_zip_path}: {cleanup_zip_error}"
                    )

                cleanup_video_dir(output_dir)

                print(
                    f" - Đã dọn sạch toàn bộ video tạm của gói "
                    f"{filename_val}."
                )

    print(
        "\n=== HOÀN TẤT QUY TRÌNH XỬ LÝ ASR BATCH CHO TẤT CẢ CÁC FILE ==="
    )


def clear_kaggle_working(output_temp_dir: str, json_final_dir: str, log_file_path: str):
    """
    Chỉ xóa dữ liệu TẠM của pipeline.

    Không xóa:
    - json_results
    - completed_zips.txt

    Điều này cho phép notebook tiếp tục từ các ZIP đã hoàn thành.
    """
    cleanup_video_dir(output_temp_dir)

    os.makedirs(json_final_dir, exist_ok=True)

    if os.path.exists(log_file_path):
        print(
            f"Giữ lại log hoàn thành: {log_file_path}"
        )
    else:
        print(
            "Chưa có completed_zips.txt, sẽ tạo khi ZIP đầu tiên hoàn thành."
        )


def start_asr_pipeline():
    print("Khởi động hệ thống xử lý Video ASR...")

    if os.path.exists("/kaggle/working"):
        csv_path = "/kaggle/input/datasets/user_name/bactch1/Batch1.csv"
        output_temp_dir = "/kaggle/working/dataset_unzipped"
        json_final_dir = "/kaggle/working/json_results"
        log_file_path = "/kaggle/working/completed_zips.txt"
    else:
        csv_path = "./Batch1.csv"
        output_temp_dir = "./dataset_unzipped"
        json_final_dir = "./json_results"
        log_file_path = "./completed_zips.txt"

    try:
        df = pd.read_csv(csv_path)
        print(
            f"-> Đã nạp CSV thành công: {len(df)} dòng dữ liệu."
        )
    except FileNotFoundError:
        print(
            f"Lỗi: Không tìm thấy file CSV tại {csv_path}."
        )
        return

    clear_kaggle_working(
        output_temp_dir=output_temp_dir,
        json_final_dir=json_final_dir,
        log_file_path=log_file_path,
    )

    asr_processing_videos(
        df=df,
        output_dir=output_temp_dir,
        json_dir=json_final_dir,
        max_hours=1,
        log_file_path=log_file_path,
    )


start_asr_pipeline()
